# 05 · Causal, prefix-LM and document masks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/architecture/05_attention_masks.ipynb) &nbsp; 📖 Concept deck: [Decoder-only, and why it won](https://nilesh-salpe.github.io/visual-ai/architecture/06-decoder-only/) &nbsp; ⏱️ Suggested time: **25 min**

**You will build:** the masks that define decoder-only, prefix-LM and packed-document training, plus a proof that a document mask makes a packed batch identical to running each document separately.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def _attn(x, mask):
    s = x @ x.T / np.sqrt(x.shape[1])
    return softmax(np.where(mask, s, -np.inf)) @ x

## Task 1: Causal mask

Return a boolean [T, T] matrix `M` with `M[i, j] = True` when query *i* may attend to key *j*. Causal (decoder-only): $j \le i$.

In [ ]:
def causal_mask(T):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_causal_mask(f):
    M = f(5)
    assert M.dtype == bool and M.shape == (5, 5), "return a boolean [T, T] array"
    assert M[3, 3] and M[3, 0] and not M[0, 3], "row i may see columns 0..i only"
    assert M.sum() == 15, "a causal 5x5 mask has 15 allowed entries"

check("causal_mask", _test_causal_mask, causal_mask)

## Task 2: Prefix-LM (non-causal decoder) mask

The first `P` tokens (the "prefix" or prompt) see **each other bidirectionally**. Later tokens are causal and see the whole prefix plus earlier targets.
This is the "non-causal decoder" in Wang et al. (2022). It sits between an encoder-decoder and a pure causal decoder.

In [ ]:
def prefix_lm_mask(T, P):
    # TODO: start from causal, then allow every pair (i, j) with i < P and j < P
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_prefix_lm_mask(f):
    M = f(6, 3)
    assert M[0, 2] and M[1, 2], "prefix tokens must see later prefix tokens"
    assert not M[2, 3], "prefix tokens must NOT see targets"
    assert M[5, 0] and M[5, 4] and not M[4, 5], "targets are causal"
    assert np.array_equal(f(6, 0), np.tril(np.ones((6, 6), bool))), "P = 0 is plain causal"

check("prefix_lm_mask", _test_prefix_lm_mask, prefix_lm_mask)

## Task 3: Packed documents: the document mask

Pretraining packs several documents into one long sequence so no tokens are wasted on padding. Without a mask, a token can attend into the *previous, unrelated* document.
Llama 3 (§3.2): *"we use an attention mask that prevents self-attention between different documents within the same sequence."*
Given `doc_ids` (e.g. `[0,0,0,1,1,2,2,2]`), allow `(i, j)` only if `j <= i` **and** `doc_ids[i] == doc_ids[j]`.

In [ ]:
def document_mask(doc_ids):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_document_mask(f):
    ids = np.array([0, 0, 0, 1, 1, 2, 2, 2]); M = f(ids)
    assert M.shape == (8, 8) and M.dtype == bool
    assert not M[3, 2], "first token of doc 1 must not see doc 0"
    assert M[4, 3] and not M[3, 4], "causal inside a document"
    x = np.random.default_rng(1).normal(size=(8, 4))
    packed = _attn(x, M)
    for doc in range(3):
        sel = ids == doc; n = sel.sum()
        alone = _attn(x[sel], np.tril(np.ones((n, n), bool)))
        assert np.allclose(packed[sel], alone), f"doc {doc}: packed output must equal running the document alone"

check("document_mask", _test_document_mask, document_mask)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Causal mask
def causal_mask_solution(T):
    return np.tril(np.ones((T, T), dtype=bool))

check("causal_mask (reference)", _test_causal_mask, causal_mask_solution)

In [ ]:
# @title Solution for Task 2: Prefix-LM (non-causal decoder) mask
def prefix_lm_mask_solution(T, P):
    M = np.tril(np.ones((T, T), dtype=bool))
    M[:P, :P] = True
    return M

check("prefix_lm_mask (reference)", _test_prefix_lm_mask, prefix_lm_mask_solution)

In [ ]:
# @title Solution for Task 3: Packed documents: the document mask
def document_mask_solution(doc_ids):
    ids = np.asarray(doc_ids)
    return np.tril(np.ones((len(ids), len(ids)), dtype=bool)) & (ids[:, None] == ids[None, :])

check("document_mask (reference)", _test_document_mask, document_mask_solution)

---
## Experiments
**How much does packing without a mask leak?** Pack 4 documents of 64 tokens and measure the fraction of attention weight that lands in *other* documents when only a causal mask is used. Llama 3 reports the document mask had limited impact in standard pretraining but mattered for continued pretraining on very long sequences.

In [ ]:
r = np.random.default_rng(0); ids = np.repeat(np.arange(4), 64); x = r.normal(size=(256, 32))
s = x @ x.T / np.sqrt(32); P = softmax(np.where(causal_mask_solution(256), s, -np.inf))
cross = (P * (ids[:, None] != ids[None, :])).sum(1)
for doc in range(4):
    print(f"doc {doc}: mean attention weight leaking into other documents = {cross[ids == doc].mean():.2f}")
print(f"cost: a causal mask has {causal_mask_solution(256).sum():,} allowed pairs; the document mask has {document_mask_solution(ids).sum():,}")

---
## Takeaways
* The architecture families differ mostly in **the mask**: causal (GPT, Llama), prefix-LM (non-causal decoder), encoder-decoder (T5, the 2017 Transformer).
* Causal decoders train on every token of every sequence and serve with a KV cache. Wang et al. (2022) found they generalize zero-shot best after pure self-supervised pretraining.
* Packing plus a document mask gives full-length batches with no cross-document leakage.